# Does PER3 genotype shift the timing of clock-gene expression in human blood?

**Research question.** Does PER3 genotype (4/4 vs 5/5) shift the time of day at which the clock genes PER2, PER3 and NR1D1 reach peak expression in human blood?

Group project notebook (GEO GSE39445). Generated by `scripts/build_notebook.py` from `notebooks/stages/`. Do not edit it by hand; edit the stage notebooks.

Run the first cell (Colab bootstrap), then *Runtime → Run all*.

**Stages:** 1 Question & objectives · 2 Data acquisition · 3 Preprocessing & QC · 4 EDA · 5 Core analysis · 6 Biological interpretation · 7 Machine learning · 8 Conclusions

In [ ]:
# --- Colab bootstrap: in Colab, clone + install the repo; locally, do nothing ---
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/CKmellow/per3-genotype-clock-gene-timing.git"
BRANCH = "main"  # change to your branch name to test it in Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo = Path("/content/per3-genotype-clock-gene-timing")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    src = str(repo / "src")
else:  # local: find the repo root (works from notebooks/ or notebooks/stages/)
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), here)
    src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)  # makes `import per3clock` work without a kernel restart

from per3clock import config  # noqa: E402

config.set_seeds()
config.ensure_dirs()
print("per3clock root:", config.ROOT, "| Colab:", IN_COLAB)

# Stage 1 · Research question and objectives

**Research question.** Does PER3 genotype (4/4 vs 5/5) shift the time of day at which the clock genes PER2, PER3 and NR1D1 reach peak expression in human blood?

**Hypothesis.** PER3 5/5 ("early bird") carriers reach peak clock-gene expression earlier in the day than 4/4 ("night owl") carriers.

**Objectives**
1. To estimate the time of peak expression of PER2, PER3 and NR1D1 in each participant.
2. To compare peak timing between the PER3 4/4 and 5/5 genotype groups and estimate the size of any shift.
3. To determine whether any shift reflects a change in the whole internal clock, using melatonin-based circadian phase.
4. To determine whether the results support the hypothesis.

## Background

**PER3 length polymorphism and chronotype.** *PER3* (Period 3) carries a variable-number tandem repeat (VNTR) present as 4 or 5 copies, giving 4/4, 4/5 and 5/5 genotypes. Archer et al. (2003) found that the polymorphism correlated significantly with extreme diurnal preference: *"the longer allele associating with morningness and the shorter allele with eveningness"*. The shorter allele was also strongly associated with delayed sleep phase syndrome. This is why 5/5 carriers are called "early birds" and 4/4 carriers "night owls".

**The molecular clock.** In each cell, CLOCK and ARNTL (BMAL1) drive transcription of the *PER* and *CRY* repressors and of *NR1D1/NR1D2* (REV-ERBα/β). These feed back on the loop and give ~24 h oscillations. Peripheral blood cells keep such rhythms, so the timing of peak expression (the **acrophase**) of PER2, PER3 and NR1D1 can be read from repeated blood samples.

**Dataset.** Möller-Levet et al. (2013) collected 10 whole-blood RNA samples from each of 26 participants during ~40 h of total sleep deprivation. This followed one week of sufficient sleep (control, ~8.5 h) and, in a crossover, one week of insufficient sleep (~5.7 h). They reported 1,855 genes with a circadian expression profile in the control condition, and listed PER1–3, CLOCK, NR1D1 and NR1D2 among the circadian-related genes affected by insufficient sleep. Participants were homozygous for the PER3 VNTR (12 with the shorter allele). The paper does not analyse peak timing by PER3 genotype, which is the gap this project addresses. Each sample also carries a **melatonin-based circadian phase**, so we can separate a shift in clock-gene timing from a shift in the whole internal clock (objective 3).

**Design choices.** The main analysis uses the sufficient-sleep ("Sleep Extension") condition. "Sleep Restriction" is used only as a sensitivity analysis. Primary genes are PER2, PER3 and NR1D1. CLOCK, CRY1, ARNTL, PER1 and NR1D2 are secondary.

In [ ]:
print("Primary genes:  ", ", ".join(config.GENES_PRIMARY))
print("Secondary genes:", ", ".join(config.GENES_SECONDARY))
print("Main condition: ", config.MAIN_CONDITION, "| sensitivity:", config.SENSITIVITY_CONDITION)
print("Period (h):     ", config.PERIOD_H, "| seed:", config.SEED)

## References
- Archer SN, Robilliard DL, Skene DJ, Smits M, Williams A, Arendt J, von Schantz M (2003). A length polymorphism in the circadian clock gene *Per3* is linked to delayed sleep phase syndrome and extreme diurnal preference. *Sleep* 26(4):413–415. doi:10.1093/sleep/26.4.413. PMID 12841365.
- Möller-Levet CS, Archer SN, Bucca G, et al. (2013). Effects of insufficient sleep on circadian rhythmicity and expression amplitude of the human blood transcriptome. *PNAS* 110(12):E1132–E1141. doi:10.1073/pnas.1217154110. PMID 23440187.
- GEO accession GSE39445. https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE39445

### Interpretation
The question is specific (three named genes, one genetic contrast, one outcome: peak time), biological, and testable with public data. GSE39445 has repeated blood samples across the 24-h cycle in genotyped participants, plus melatonin-based phase. Objectives 1–2 test the hypothesis directly. Objective 3 asks whether any difference reflects the whole internal clock (a shift relative to clock time that disappears relative to melatonin phase) or clock-gene timing specifically. Objective 4 brings the evidence together.

# Stage 2 · Data acquisition

All data come from GEO series **GSE39445** (platform **GPL15331**, Agilent-026817 University_Surrey_HumanSleep_44k_v1). Small source files are committed in `data/raw/`. The ~100 MB whole-transcriptome series matrix is downloaded on first use and cached in `data/external/`. Everything is loaded through `per3clock.data`, the same functions used by every later stage. The full data dictionary is in `data/README.md`, and the verification report is `results/tables/data_verification.md`.

| File | What it is |
|---|---|
| `data/raw/GSE39445_sample_metadata.csv` | one row per sample: subject, genotype, condition, time awake, clock time, circadian phase |
| `data/raw/GSE39445_clock_genes.csv` | the same samples × 14 probes for 8 clock genes (log2 expression) |
| `data/raw/GPL15331_probe_annotation.csv` | probe ID → gene symbol, name, Ensembl ID, location |
| `data/raw/GSE39445.txt` | GEO SOFT sample headers (provenance) |
| `data/external/GSE39445_series_matrix.txt.gz` | whole transcriptome: 43,758 probes × 438 samples (downloaded) |

In [ ]:
import pandas as pd
from IPython.display import display

from per3clock import data

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

## 2.1 Sample metadata

`load_metadata()` reads the metadata CSV, recovers the 11 samples whose subject and genotype were blank (from the sample title and the subject's other samples), parses `clock_time` into decimal hours, and adds `condition_code`, `timepoint` and `is_recovered`.

In [ ]:
meta = data.load_metadata()
print(meta.shape)
display(meta.head())
display(meta.dtypes.to_frame("dtype").T)

| Column | Meaning | Units / values |
|---|---|---|
| `sample_id` | GEO sample accession | GSM… |
| `title` | GEO sample title | `Subject_<id>_<SE/SR>_T<n>` |
| `subject` | participant ID | 26 IDs (AF0004…AF0335) |
| `per3_genotype` | PER3 VNTR genotype | `4/4`, `5/5` |
| `sleep_condition` | prior week of sleep | `Sleep Extension` (sufficient, control), `Sleep Restriction` (6 h/night) |
| `condition_code` | short code | `SE`, `SR` |
| `timepoint`, `timepoint_n` | sample number within the 40-h wake period | T1–T10 / 1–10 |
| `hours_awake` | time since waking | h (7.5, 10.5, …, 34.5) |
| `clock_time` | clock time of blood draw | `H:MM` (no date) |
| `clock_time_h` | same, decimal | h in [0, 24) |
| `circadian_phase_h` | time relative to the participant's plasma-melatonin rhythm (GEO) | h, 3-h steps from −15 to 15 |
| `is_recovered` | subject/genotype recovered from title | bool |

In [ ]:
display(meta.loc[meta.is_recovered, ["sample_id", "title", "subject", "per3_genotype", "condition_code", "timepoint"]])

## 2.2 Clock-gene expression

`load_clock_genes()` returns a long table (sample × gene). **Default: one probe per gene**, the probe with the highest mean expression across all 438 samples (`select_probes`, `collapse="max_mean"`). Averaging would mix in near-background probes. For example, two of PER3's three probes have mean log2 3.9 and 4.2. The rule uses only each probe's overall mean, never time, rhythmicity or genotype, so it cannot bias the later tests. `collapse="mean"` (average probes) and `collapse=None` (probe level) remain available.

In [ ]:
clock = data.load_clock_genes()                  # default: selected probe per gene
probes = data.load_clock_genes(collapse=None)    # every probe
print("long table:", clock.shape, "| probe-level:", probes.shape)
display(clock[["sample_id", "gene", "probe_id", "expression", "n_probes", "subject", "per3_genotype", "condition_code", "timepoint", "clock_time_h"]].head(8))

selection = data.clock_probe_selection()         # also saved as results/tables/selected_probes.csv
display(selection)

`expression` is the GEO quantile-normalised **log2** signal (unitless). Each gene has 1–3 Agilent probes, and their IDs match the GPL15331 annotation and the series matrix. The selected probes are PER2 16821, PER3 25341, NR1D1 33493, CRY1 22736 and NR1D2 38683; CLOCK, ARNTL and PER1 have a single probe. PER3 remains the weakest primary gene (mean log2 5.5 against an array median of 6.7).

## 2.3 Probe annotation (GPL15331)

In [ ]:
annot = data.load_probe_annotation()
print(annot.shape, "| probes with a gene symbol:", annot.GENE_SYMBOL.notna().sum())
display(annot.head(3))
display(annot.dropna(subset=["GENE_SYMBOL"]).head(3))

| Column | Meaning |
|---|---|
| `ID` (index) | integer probe ID (= series-matrix `ID_REF`) |
| `PROBE_NAME` | Agilent probe name, or a control name (e.g. `GE_BrightCorner`) |
| `GENE_SYMBOL`, `GENE_NAME` | HGNC symbol and name (blank for controls) |
| `ENSEMBL_ID` | Ensembl identifier |
| `CHROMOSOMAL_LOCATION` | probe coordinates |

## 2.4 GEO SOFT sample headers (provenance)

In [ ]:
soft = data.load_soft_samples()
print(soft.shape)
display(soft.head(3))
print("Samples storing genotype under 'genotype' instead of 'genotype/variation':", soft["genotype"].notna().sum())

## 2.5 Whole-transcriptome series matrix

Downloaded once (~100 MB, about 30 s in Colab) to `data/external/` and cached. `load_expression_matrix()` checks that there are 438 samples and a plausible number of probes. `collapse_probes_to_genes()` makes a genes × samples matrix, keeping by default the same outcome-blind choice of one probe per gene (highest mean across all samples).

In [ ]:
expr, chars = data.load_expression_matrix(with_characteristics=True)
print("probes x samples:", expr.shape)
print(f"log2 range {expr.values.min():.2f}-{expr.values.max():.2f}, median {pd.Series(expr.values.ravel()).median():.2f}, NaN: {int(expr.isna().sum().sum())}")
genes = data.collapse_probes_to_genes(expr, annot)
print("genes x samples:", genes.shape)
display(chars.head(3))

In [ ]:
# Check: the clock-gene CSV equals the series matrix for all 14 probes x 438 samples
wide = probes.pivot(index="sample_id", columns="probe_id", values="expression")
diff = (expr.loc[wide.columns, wide.index].T - wide).abs().to_numpy().max()
print("max |clock CSV - series matrix| =", diff)
assert diff < 1e-6

## 2.6 Study summary and the shared analysis sets

`analysis_samples()` defines the sample set every later stage uses (issue #10):
- **default**: Sleep Extension, subjects with ≥6 of 10 timepoints (main analysis);
- `strict=True`: the original authors' rule (drop series with 2 consecutive or >2 missing timepoints), as a sensitivity analysis;
- `for_phase_analysis=True`: also drops AF0182 in SE, whose melatonin-phase labels are inconsistent. Use it whenever time is `circadian_phase_h`. AF0182 stays in clock-time analyses because its clock times are real.

In [ ]:
summary = (
    meta.groupby(["per3_genotype", "sleep_condition"])
    .agg(subjects=("subject", "nunique"), samples=("sample_id", "size"))
    .unstack("sleep_condition")
)
display(summary)

tp = meta.pivot_table(index=["per3_genotype", "subject"], columns="condition_code", values="sample_id", aggfunc="count", fill_value=0)
display(tp)

variants = {
    "default (main)": dict(),
    "strict=True (sensitivity)": dict(strict=True),
    "for_phase_analysis=True": dict(for_phase_analysis=True),
    "Sleep Restriction (sensitivity)": dict(condition="Sleep Restriction"),
}
rows = []
for label, kw in variants.items():
    s, excl = data.analysis_samples(**kw, metadata=meta, return_exclusions=True)
    gt = s.drop_duplicates("subject").per3_genotype.value_counts()
    rows.append({"set": label, "subjects": s.subject.nunique(), "4/4": gt.get("4/4", 0),
                 "5/5": gt.get("5/5", 0), "samples": len(s), "excluded": ", ".join(excl.subject)})
display(pd.DataFrame(rows))

_, excl = data.analysis_samples(strict=True, for_phase_analysis=True, metadata=meta, return_exclusions=True)
print("Why subjects are excluded (Sleep Extension, strict + phase):")
display(excl)

### Interpretation

- **Size and design.** 438 blood samples from 26 participants: 12 PER3 4/4 and 14 PER3 5/5, all homozygous, so the comparison is a clean two-group contrast. 221 samples are from the sufficient-sleep (SE) condition and 217 from sleep restriction (SR). Samples were taken every 3 h from 7.5 to 34.5 h awake, which covers more than one full 24-h cycle.
- **Data integrity.** After recovery, every sample has a subject and genotype. The 11 recovered genotypes agree with GEO's own (differently named) genotype field. The clock-gene CSV is identical to the series matrix (max difference 0). The metadata agree with GEO on every field.
- **Shared analysis sets.** Main analysis: 24 participants (11 × 4/4, 13 × 5/5), 221 SE samples. Strict sensitivity set: 20 participants (9 × 4/4, 11 × 5/5), 193 samples. Melatonin-phase analyses: 23 participants (AF0182 dropped), 214 samples. AF0079 and AF0091 have only SR samples, and AF0186 and AF0295 have only SE samples. The genotype groups are small, so later stages should report effect sizes with confidence intervals, not only p-values.
- **One probe per gene.** Using the most highly expressed probe avoids diluting signals with near-background probes (two of PER3's three). The choice is outcome-blind.
- **Things later stages must handle** (details in `results/tables/data_verification.md`):
  1. AF0182 SE T4/T5 have identical circadian phase with irregular spacing, probably an annotation error. AF0182 is excluded from phase analyses only.
  2. PER3 is weakly expressed on this array, so Part 2 should confirm it is above background.
  3. `circadian_phase_h` is binned in 3-h steps. The implied phase-0 clock time averages ~04:10, close to the published melatonin midpoint, but the exact definition is unconfirmed.
- The whole-transcriptome matrix (43,758 probes, ~19.5k genes) is available for the genome-wide rhythmicity scan, the enrichment analysis and the ML stage.

**Can these data answer the question?** Yes, with caveats. The data have everything the question needs: PER3-genotyped participants in two homozygous groups, repeated blood samples spanning more than 24 h (so a peak time can be estimated per person), the three target genes measured on the array, and a melatonin-based phase for objective 3. The limits are a small sample (~12 per genotype, so only fairly large shifts are detectable), 3-hourly sampling and 3-h phase bins (coarse timing), and a single cohort with no independent replication set. These limit precision and power, not validity, and stage 8 must state them.

# Stage 3 · Preprocessing and quality control

*Owner: Part 2 (see issue #2). Skeleton: replace the TODO cells, keep the headings.*

## Minimum evidence (lecturer's brief)
*Brief, stage 3: "Clean, filter, normalize or align the data; check it against biology you know must hold." Minimum evidence: "Before/after counts, a comment on anything suspicious."*

- [ ] Clean / filter / normalize: the data are already quantile-normalised by GEO (limma). Show that array distributions are comparable, and document every filter you add
- [ ] **Before/after counts** for every step (samples, subjects per genotype, probes/genes)
- [ ] **Check against biology you know must hold** (e.g. core clock genes expressed above background in blood, which matters most for PER3, the weakest; clock genes vary with time of day in the expected order; sex-marker genes consistent within each subject)
- [ ] **A comment on anything suspicious** (outlier arrays, AF0182 SE phase labels, recovered samples, late draws)
- [ ] Review the shared defaults (`analysis_samples()`, one-probe-per-gene) and say whether QC supports them
- [ ] Short interpretation of your own results
- [ ] Report what the data does **and does not** show. A null, negative or inconclusive result is a valid result: report it with its uncertainty.

## Contract
**Inputs:** `data.load_metadata()`, `data.load_expression_matrix()`, `data.load_probe_annotation()`, `data.collapse_probes_to_genes()` (default: one probe per gene, highest mean), `data.analysis_samples(..., return_exclusions=True)`, `results/tables/selected_probes.csv`, `sample_anomalies.csv`, `timepoints_per_subject.csv`.

**Shared defaults (issue #10, `data/README.md` → *Shared defaults*):** main set = SE, ≥6 timepoints (24 subjects, 221 samples). `strict=True` = authors' rule (20 subjects, sensitivity). `for_phase_analysis=True` drops AF0182 SE (23 subjects). These are the single source of truth: if QC justifies a change, change `analysis_samples()` in its own PR (tag Part 1) rather than saving a separate sample list.

**Implement in `per3clock.qc`:** `filter_low_expression`, `sample_qc_table`, `apply_qc`, `qc_counts`. Remove the `xfail` markers in `tests/test_qc.py` once done.

**Outputs (names are fixed):**
- `results/tables/qc_counts.csv`: before/after counts per step
- `results/tables/qc_exclusions.csv`: `sample_id, subject, sleep_condition, reason`
- figures: `qc_expression_density.png`, `qc_sample_correlation.png`, `qc_biology_checks.png` (via `plotting.save_fig`)

In [ ]:
import pandas as pd
from IPython.display import display

from per3clock import data, plotting

plotting.set_style()

## 3.1 Load data

In [ ]:
# TODO(Part 2): load metadata and expression matrix; record starting counts

## 3.2 Sample-level QC

In [ ]:
# TODO(Part 2): sample_qc_table(); flag outliers; comment on AF0182 SE T4/T5 (excluded from phase analyses by default)

## 3.3 Probe/gene-level filtering

In [ ]:
# TODO(Part 2): filter_low_expression(); justify thresholds with a density plot

## 3.4 Time-series completeness

In [ ]:
# TODO(Part 2): print analysis_samples(return_exclusions=True) for default and strict=True; check the defaults; apply_qc()

## 3.5 Biology sanity checks

In [ ]:
# TODO(Part 2): biology that must hold, e.g. clock genes above background (PER3 weakest); PER/NR1D1 vary with time of day; sex markers consistent per subject

## 3.6 Before/after table and outputs

In [ ]:
# TODO(Part 2): qc_counts(); save qc_counts.csv and qc_exclusions.csv

## Interpretation

_TODO(Part 2): what was removed and why (with counts); is anything suspicious left; does the data agree with clock biology that must hold; do the shared defaults stand?_

# Stage 4 · Exploratory data analysis

*Owner: Part 2 (see issue #2). Skeleton: replace the TODO cells, keep the headings.*

## Minimum evidence (lecturer's brief)
*Brief, stage 4: "Describe and visualize the data; look for patterns, outliers." Minimum evidence: "at least three informative plots, interpretation."*

- [ ] **At least three informative plots** (more is fine; uninformative plots don't count)
- [ ] **An interpretation for each plot**: what pattern or outlier it shows and why it matters for the question
- [ ] Look explicitly for patterns (time of day, subject, genotype, condition) and outliers
- [ ] Short interpretation of your own results
- [ ] Report what the data does **and does not** show. A null, negative or inconclusive result is a valid result: report it with its uncertainty.

## Contract
**Inputs:** `data.analysis_samples()` (shared default; use `for_phase_analysis=True` for plots against `circadian_phase_h`), `data.load_clock_genes()` (one probe per gene), expression matrix via `data.collapse_probes_to_genes(data.load_expression_matrix())`.

**Implement in `per3clock.eda`:** `plot_clock_gene_profiles`, `pca_samples`, `plot_pca`, `plot_sample_correlation`.

**Outputs:** figures `eda_clock_gene_profiles.png`, `eda_pca_subject.png`, `eda_pca_timepoint.png`, `eda_sample_correlation.png` (via `plotting.save_fig`).

In [ ]:
import pandas as pd
from IPython.display import display

from per3clock import data, plotting

plotting.set_style()

## 4.1 Clock-gene profiles by genotype

In [ ]:
# TODO(Part 2): plot_clock_gene_profiles() for PER2, PER3, NR1D1 vs clock time and vs circadian phase

## 4.2 PCA

In [ ]:
# TODO(Part 2): pca_samples(); colour by subject, condition, timepoint, genotype

## 4.3 Sample correlation

In [ ]:
# TODO(Part 2): plot_sample_correlation()

## Interpretation

_TODO(Part 2): What dominates variation (subject? time of day? condition?); do the clock genes look rhythmic; any visible genotype difference?_

# Stage 5 · Core analysis: peak timing by PER3 genotype

*Owner: Part 3 (see issue #3). Skeleton: replace the TODO cells, keep the headings.*

## Minimum evidence (lecturer's brief)
*Brief, stage 5: "Apply the methods that directly answer your question." Minimum evidence: "Results table, key figures, statistical evidence" (example: "with FDR correction").*

- [ ] Methods that directly answer objectives 1–3 (per-subject peak times; 4/4 vs 5/5 comparison; melatonin-relative phase)
- [ ] **Results table** (estimates, effect sizes in hours with 95% CIs, test statistics, p-values)
- [ ] **Key figures** (e.g. acrophase rose plots, forest plot of shifts)
- [ ] **Statistical evidence** for each objective, with assumptions checked
- [ ] **FDR correction** (Benjamini–Hochberg) wherever many tests are run: the genome-wide rhythmicity scan, and across the primary + secondary genes
- [ ] Sensitivity analyses: `strict=True` sample set; Sleep Restriction; probe-mean (`collapse="mean"`) vs selected probe
- [ ] Short interpretation of your own results
- [ ] Report what the data does **and does not** show. A null, negative or inconclusive result is a valid result: report it with its uncertainty. With ~12 subjects per genotype, a non-significant shift is *inconclusive*, not proof of no shift. Give the CI.

## Contract
**Sample sets (shared defaults, `data/README.md`):**
- objectives 1–2 (clock time): `data.analysis_samples()` → 24 subjects (11 × 4/4, 13 × 5/5), 221 samples
- objective 3 (melatonin phase, `time_col="circadian_phase_h"`): `data.analysis_samples(for_phase_analysis=True)` → 23 subjects (AF0182 SE dropped)
- sensitivity: `strict=True` (20 subjects) and `condition="Sleep Restriction"`

**Inputs:** `data.load_clock_genes()` (one probe per gene; `collapse="mean"` for the probe sensitivity check), expression matrix via `data.collapse_probes_to_genes(data.load_expression_matrix())`.

**Implement in `per3clock.rhythm`:** `fit_subject_cosinors`, `rhythmicity_scan` (`cosinor_fit` exists: extend and validate it). **In `per3clock.circstats`:** `circular_mean_h`, `circular_diff_h`, `watson_williams_test`, `permutation_test_acrophase`, `bootstrap_ci_diff_h`.

**Outputs:**
- `results/tables/acrophase_by_subject.csv`: subject × gene × condition × time base (clock time / circadian phase)
- `results/tables/acrophase_genotype_tests.csv`: gene, time base, condition, sample set, mean acrophase 4/4 and 5/5, shift (h, 5/5 − 4/4), 95% CI, permutation p, Watson–Williams p, BH q
- `results/tables/rhythmic_genes_SE.csv`: genome-wide scan with q-values (**input to Part 4**)
- figures: `acrophase_rose_PER2.png` / `_PER3` / `_NR1D1`, `acrophase_shift_forest.png`

In [ ]:
import pandas as pd
from IPython.display import display

from per3clock import data, plotting

plotting.set_style()

## 5.1 Per-subject cosinor fits (objective 1)

In [ ]:
# TODO(Part 3): fit_subject_cosinors(time_col='clock_time_h') on SE; report fit quality

## 5.2 Genotype comparison in clock time (objective 2)

In [ ]:
# TODO(Part 3): permutation test + Watson–Williams per gene; shift in hours with bootstrap CI; BH across genes

## 5.3 Melatonin-relative phase (objective 3)

In [ ]:
# TODO(Part 3): repeat with time_col='circadian_phase_h' on analysis_samples(for_phase_analysis=True); compare shifts with 5.2

## 5.4 Sensitivity: Sleep Restriction

In [ ]:
# TODO(Part 3): repeat 5.1-5.3 with strict=True, on Sleep Restriction, and with collapse='mean' probes

## 5.5 Genome-wide rhythmicity scan

In [ ]:
# TODO(Part 3): rhythmicity_scan() with BH FDR; save rhythmic_genes_SE.csv

## Interpretation

_TODO(Part 3): for each objective (1–3): the estimate, its uncertainty and what it means. Do 5/5 carriers peak earlier, by how many hours (95% CI), and does the shift remain relative to melatonin phase? Do the sensitivity analyses agree? If the result is null or inconclusive, say so plainly._

# Stage 6 · Biological interpretation

*Owner: Part 4 (see issue #4). Skeleton: replace the TODO cells, keep the headings.*

## Minimum evidence (lecturer's brief)
*Brief, stage 6: "Explain what the results mean biologically." Minimum evidence: "At least one interpretation method, linked back to the question."*

- [ ] **At least one interpretation method** (we plan GO/pathway enrichment, a STRING/NetworkX network and a literature comparison)
- [ ] Each method **explicitly linked back to the research question** (PER3 genotype and the timing of PER2/PER3/NR1D1 peaks)
- [ ] Every citation verified (DOI or PubMed); no invented references
- [ ] Short interpretation of your own results
- [ ] Report what the data does **and does not** show. A null, negative or inconclusive result is a valid result: report it with its uncertainty.

## Contract
**Inputs:** `results/tables/rhythmic_genes_SE.csv` and `acrophase_genotype_tests.csv` (stage 5). While they don't exist, develop with the 8 clock genes and **don't commit placeholder results**. Background for enrichment: all genes on the array after QC (`data.collapse_probes_to_genes(...)` index).

**Implement in `per3clock.interpret`:** `run_enrichment`, `fetch_string_network`, `build_network`, `network_centrality`. Cache API responses in `data/external/`.

**Outputs:** `results/tables/enrichment_rhythmic.csv`, `results/tables/string_edges.csv`, `results/tables/network_centrality.csv`; figures `enrichment_top_terms.png`, `string_network.png`.

In [ ]:
import pandas as pd
from IPython.display import display

from per3clock import data, plotting

plotting.set_style()

## 6.1 Enrichment of rhythmic genes

In [ ]:
# TODO(Part 4): run_enrichment() on q<0.05 rhythmic genes vs array background

## 6.2 STRING network of clock genes

In [ ]:
# TODO(Part 4): fetch_string_network(); build_network(); network_centrality(); plot

## 6.3 Literature comparison

In [ ]:
# TODO(Part 4): compare acrophases/shift with published values (verify every citation)

## Interpretation

_TODO(Part 4): what the rhythmic / shifted genes do, how they connect to PER3 in the clock network, how our results compare with the literature, and what that means for the research question._

# Stage 7 · Machine learning: predicting internal circadian phase

*Owner: Part 5 (see issue #5). Skeleton: replace the TODO cells, keep the headings.*

## Minimum evidence (lecturer's brief)
*Brief, stage 7: "Build a model that classifies, predicts or groups your data." Learning outcome 6: "Build and evaluate machine learning models against proper baselines."*

- [ ] A model that **predicts** each sample's internal circadian phase (`circadian_phase_h`) from expression
- [ ] **Evaluated against proper baselines** with leave-one-subject-out CV (no leakage: scaling and feature selection inside folds)
- [ ] Metric: **median absolute circular error in hours**, with the same folds for every model
- [ ] A fairer "added value" comparison: **clock time + genes vs clock time alone**
- [ ] Short interpretation of your own results
- [ ] Report what the data does **and does not** show. A null, negative or inconclusive result is a valid result: report it with its uncertainty. Matching the clock-time baseline without beating it is an acceptable, reportable result.

## Contract
**Samples:** `data.analysis_samples(for_phase_analysis=True)` (target is `circadian_phase_h`; AF0182 SE dropped; 23 subjects, 214 samples). **Features:** clock genes via `data.load_clock_genes()` (one probe per gene) and/or genes from `data.collapse_probes_to_genes(data.load_expression_matrix())`.

**Target:** `circadian_phase_h` encoded as sin/cos. **CV:** `LeaveOneGroupOut`/`GroupKFold` by `subject`. **Metric:** median absolute circular error (h).
**Baselines:** (a) circular mean phase of the training fold; (b) phase predicted from `clock_time_h` only. Optional: add PER3 genotype as a feature.

**Read this before interpreting (data caveat):** `circadian_phase_h` comes in **3-h steps** and is roughly **clock time minus a per-subject offset** (the clock time of each person's melatonin phase 0, ~02:15–07:25 in SE). So baseline (b) will be **strong**: it is only wrong by that offset. Compare the model with it honestly, using the same LOSO folds and the median absolute circular error. **Matching but not beating the clock-time baseline is an acceptable, reportable result.** The fairer question is whether expression adds information beyond clock time: compare **clock time + genes** against **clock time alone** (e.g. does adding genes reduce the per-subject offset error?). Errors below ~1.5 h are not meaningful with 3-h bins. GSE48113 has no phase labels, so there is no external validation set (`data/README.md`).

**Implement in `per3clock.ml`:** `encode_phase`, `decode_phase`, `circular_abs_error_h`, `loso_cv_predict`, `evaluate_phase_models`.

**Outputs:** `results/tables/ml_phase_prediction.csv` (one row per model, including both baselines and clock time + genes); figures `ml_predicted_vs_true_phase.png`, `ml_error_by_model.png`.

In [ ]:
import pandas as pd
from IPython.display import display

from per3clock import data, plotting

plotting.set_style()

## 7.1 Features and target

In [ ]:
# TODO(Part 5): build X (genes) and y (sin/cos of circadian_phase_h); groups = subject

## 7.2 Baselines

In [ ]:
# TODO(Part 5): mean-phase and clock-time-only baselines under LOSO

## 7.3 Model

In [ ]:
# TODO(Part 5): e.g. Ridge / ElasticNet / RandomForest in a Pipeline; tune inside folds (nested CV)

## 7.3b Does expression add information beyond clock time?

In [ ]:
# TODO(Part 5): same LOSO folds: clock_time_h only vs clock_time_h + genes; report the difference in median abs circular error (h) with a paired comparison across subjects

## 7.4 Comparison

In [ ]:
# TODO(Part 5): evaluate_phase_models(); table + figures; optional genotype feature

## Interpretation

_TODO(Part 5): does expression predict internal phase better than the baselines, and does it add anything beyond clock time? Which genes carry the signal? If the model only matches the clock-time baseline, say so: that is a valid finding._

# Stage 8 · Conclusions

*Owner: Part 1 (see issue #6). Skeleton: replace the TODO cells, keep the headings.*

## Minimum evidence (lecturer's brief)
*Brief, stage 8: "Answer the research question and state the limitations." Minimum evidence: "One point per objective, plus future steps. Upload to GitHub the code." Learning outcome 7: "Communicate results honestly, including negative or inconclusive findings."*

- [ ] **Answer the research question** in one or two sentences
- [ ] **One point per objective** (1–4), each citing the evidence (table/figure) it rests on
- [ ] **Limitations**
- [ ] **Future steps**
- [ ] Negative or inconclusive findings stated honestly (no overclaiming from small n)
- [ ] Code on GitHub; the final notebook runs top-to-bottom in Google Colab **with outputs saved**

## Contract
**Inputs:** tables in `results/tables/` from stages 3–7 (`qc_counts.csv`, `acrophase_genotype_tests.csv`, `rhythmic_genes_SE.csv`, `enrichment_rhythmic.csv`, `ml_phase_prediction.csv`). Written last, by Part 1, once results exist (issue #6).

In [ ]:
import pandas as pd
from IPython.display import display

from per3clock import data, plotting

plotting.set_style()

## 8.1 Key numbers

In [ ]:
# TODO(Part 1): load acrophase_genotype_tests.csv, ml_phase_prediction.csv, qc_counts.csv and print the headline results

## Interpretation

_TODO(Part 1): Objective 1: … · Objective 2: … · Objective 3: … · Objective 4 (hypothesis supported?): … · Limitations: small n (24 SE subjects), 3-h sampling and binned phase, homozygotes only, blood not central clock, single dataset · Future steps: …_